# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz27aW3/UrsOx0SrYyKzmPptqqs0nstJl1Em+Stnev6+FAIigx5qt8yFYy2b91v99ftuccACT4kp3cXE8i8QEcAAfnjQPIsqwXaZnmCxbAl1047OhnFvMo4uzeCbN/rZJ9xZN7J0fH7Ht2HO/ofSluyirnePkkEokvcncy+a1ImS9YLtZVXqTFYsLYt+xt6qcFi+B/nPoCv/+qhM8Ttub5huM7kTBo65fz3xjHi514z+xM5EEYsReP//b8xSv2PUBi7PHZ21fOlPEiZEnKtnzPRAHlOEIoc8GqhNMQ2B7BFOGmCuGFcKkbZ9BQJqBXrBAsgy5i64lq+/XjF8xe8/VWYP8zvgkTgAqXZ2FS3TgsEnsYY0q9wBHwfL0Nd7LrGc95BONaMAU0h46osbKS5z50RWwqqF7InpxGrMz5ir+D1yLOUqjOdhxBfe+L3ffFNma2HxbrFB9Bzxw1AIki6n1aio1g6zTBptY88QGP/6kngtlPodx3VBXK5IAZkVDXdVeppokithEJIHIdpkkzGw3eclFUUUlzBc1vKhwUFUSMS7TJrjIYDUxMsUXcRXo2GrS9q4pSFIipTJRhjkXisIhVQVGwMClKnsA/kboTy7ImISKoxA4DTeXQVzeogPCgKEzQOtDvN+v6KkpX+hp7EoX1bZjqq3dFmujrtNBXudBXxbYqw6i+q1aA8rUo6pLlNhfcD5NN/SCM68rvwwxIV0z0fVLF2R67m2STIE9jdv78jKl3z2O+EVP59SorJrKAy5phB+FmMvn19PUpW0JX3YyXW9cP84THwtb3fFXgt+152LDnOc4E5s87e/X0v6FW3Vn3LF1f2c7k9+enf7yBF7YFrSWlNWXWiq+v8DsSAd3n4WZbWs5kMvmKvQHqzUMgsQV0iyOlFGwTAvPAVLKUBTwqcHApy4CvgQ7KKl4BE4gIiCXfhX6aT2h8LjCz9/zF419OvfPnfzs9wy48vO/NZjP8P8G3b56f4DiPZ/cfQdO+CFjBA+FRs3YYb5wF0RIQxu88gi4BpydpDrIqfM+JGGUHccbwrV3ymCfpFLkRPgMsWqaOi4SFcMIAqgNdFJLs1tSGmg2XPlWD+JfzEIgf2q3EaZ6nuW0BIytBgKMHZBg92FH/AINY9XrKtjAsAO4W4Xuh275mP7F7xwxZhq4OtXVWQxYomGIQgsCPINFAmEKrdhwmYZwiuOzGUc1SE98C8J/Z8AR8RoObHKSNaBqI+Y2Nw3OgET2DDVgYsRr3Os32tmO+cMstEErCw8i2dc1pDcPR03D2+OXTv796UzeINVG4yplLgIhf//IESRa+HlvOcNvJTuSlrYpQiVyAFEnwtSK0dSR4UmVeWpVZVRb2lRDZ8tEMaCdMvCAXwtuslvNZQ4FPQLJyEmuGeAQ0sV0o3qGkDElF1ahTvFJyRsK9JsIy3zed9gvocwGsL3zbR4JlPg4ShZqLHzXLv0vDxJbiwX3121vv5PlrwMG3luMgknSpsABRYftOg3gcCTZBAg4ESXHlVQVyVxuW41JB0PfffntvVte+3oJ8wU4CETAb1I3tFzjziCykY6r0k4kyY0JofNC2D51LM3vmtN6oHuUxaHJh+1MWboC1hSeQHIvl27wS7QrGMMVNWMCU+aD0LBfEr9VpFf+gbC7idCfMYq1Sn48acbMWWclO6Qt0aNN6xkFpTCbrCL7ZaQJ2hZAvkeQ8D5i29Dy7EFEwRbUeLF+mCeiDKN0sszxMSmMgWMjNoINYDpGtuuWLUqxLg7eoIECAovDZfgwoWOELbKb9ptjyTIy/8eLdgZflwYojrzMOAxx/412FCZKLYpP65bt0hUzyweLrEm0wa8GASy2ReOs04nj3sV3BA2sPePIaKj0DXdW09RWobTTAgG2FHyKH2ngBNkm6CyOHzCWQfqDbQZ2twQYuFQe34QtgaewPqRdo/5GL/VEGMtzfe0APVtI6gwcPZ26vjwDEQ1o2tPwwi6suu2jCGATckiJm19wq83kpbCwPVMF9O82Ab9utOoaEGCfmmqBbbUj7DxDwsUtq3N/3cE6vFCq8LE0jKLEO3Ldko5zD/ekNeA/gjdigWbzrNL8SwP3HoMYkZVxY6pl12XQZ0BHzK5i6vOghjGSDl14pCTKRE3/0L/xJx6XmYqAuRKt3LdBmKhQzg4WbFksbyADMiqT83vCiLKKO3uN4B8K7QTaohzMQM2XtPdU+B3oo5MgQffriHc+j2osCI7zWLFKogT25rTYgeTYBXwtvW620dVmA5+DR9HmAuqYKyHjksIvLcepCzZSjZmrDsB2XBt4XvyCtc3rnhaTRRorVsMWOSrnIugVQ4UjZpr/fQYeLMrcDl2xgomoCFcjmdvS8uBylc7TQxQAPgQy1rZcpOV4VWECREOS41O7ir88WMKUd7aTsi9mkUe1AK7lvU8fazSSA7FnryXUIkoD4FIujPb6CJlZVEAjQCpvlzMHeBn2cSAWNpD6MrxU0JVnTfnif/fQTmNrOYEFlGq/G0a5GmAwWSHA+0D5YOQ0GUNijo+Tih6GwaLTDQmD+kEYqbtr9KNOSo+QoqtgWN8D8mZ2DyUAz7PQ1oR1Y55JjlIuuog7ALAv2QQJT+nzhzoOP7Jcn+NrqDy2wPhgjYEesXLiz4GNhSGKFF4I6aXS9VIZZmAlDQkzRU5mCx5SvUoMoJNPu7/lgfkstina44tpaZJyEb59F6fULXoJgSDbnADoKDVWKbQGObivvYmuoI0twAhKwOzt0rHoZpBGI7CX1tyqEh44ZSLGib5wB6dCA2lP2FXsGRtD29xNAuS/WqR8G4ZriPTLU9H/z2Q2ZzznPQhVqGRU/engu+D6gTLwAQe/82I7XHo826dKK1x3rTrH7230mPZw+wMFWRlvqs80dBEpfsLAaLehDgtGZpUkIDQ2IlDvC/2TYimBxlA29kqUgCfUaDB+w4miW1R0ZaOYDmnDzwV8VOMDlnuzZtlZ7igxoxgNd9hZrM/sBWhfw/Lu63w46nyIi/51iS9A1MHYN0qIgE9hdrOARRaHqWBt7CijCiJl0WVnJ41UIBSj6qIOPALwGpUJ2MqAAvZnJ7mg7UPXiKTn2fkvJ6iBQCiq6kXZ452JgBYydwl1XPgdBVcZV5ALBp9deGYC3viRhfaBSknxq8RXYFlswWa+6pVtiRXoCqudPAMImTwFxr8lHyk1mNueThQXZ622ya5VY1pYa9NtLE2+TVdYlOiyoUBpquV0vQNuG0zLYcsup6Y2iDatpmtzXlvsyDpw4iWgWqIqZwXBFtZL8SGpxx3XdjtwZlCod10ka44aGGDYbR1Xx2J+1VdX9Iz8sj3bHR7MjQgAIgL5D/RnyRaLgNsE1QCItuuzSWHt2Pmtu5KxMgc2JWwfmpeX03nEChhAKT8nD6ZMaeMw4knGyI8/4M0YHgBXt7XgeUsSpKHnxGcSnfPM7DR/8k0+nwEEijHdflgobhByixM602C2SA+k0MjFOf9Y+c87as3WINm+ftn/vpA1NWZ/O/4XZas3TFF2rqkAlW6SggOe0GjQyezJoVE+JvB2cj5a2K8XNkAl9jtV7hjP1uKIok31h4IIaM6XoZUetCMAPuM0OWDDD9azLnhddoD9OKxbFAbwFQ0RFvWf2h2+o9W+wM/UlQZQd+kZaNN98HKC0g3bv6IteCG8YoYc9ix7RDYvclusx7KvebtobVGTJqVMYWhzsEq2KfFEMDA7SmRwGryOh0N9hpx7c5qsv5Y7cRZkfDuGaplvNn2j79Xm0iX8WApiIQqbgWobvpZOMRiQ8NEN9vQYurDQI0FuxLhe3dFT7cGToe+us8lTVjjOHXHOL3aOyH54/buOL2WuQYDmG6Pfgp6CD0SyEO1ZLnCFGhizlfqAto3WulvyfdtWV+cAZDL9l5jSQBK2fmOboiAvslqltAcosZzjK2zLo6M01z+MqswejMS9UJkgEqlCGYm4Nq+jaXizQ+a4dVdUMvm77mb8lHB047mOOQRKCjQ5enUzQWLB1GmdhJN3QK5EnMOU4U09/O3nM9kAA4aYZDQVfo02ah2Usw6/r6uTlS5mIAjCyHHqd62QHjrkcHOO2mMkSpeu7eYsqWNNyDFpGiUk6VFgpPe1YSVeQvMyw8PiOw/CAIu0OMUhn/1bXqx/wHUpcOMn5dZvtY4An120Tca0WXKfgS8+Ppww+HLieTZn65zi9tcEarIsfdhh3irgiisKsELY9/+HhlN0HMPfuwcXxbOZQ/C9a2nA9ZfMf8OMBfBw/eNBtCKgWmDPZRAjnR6g+/xEBzRHiDw2gh/AQV4DnDwfBoMspsf4LZc9gvNIXu3AtlhbOg+WAv5+AgQnCDUR/J9BKQU9ZPUwCkaNeIMlkQ/vyOa/KdM2L0pbgpswvQcUt5UsUXOX8oTMcGLNp7X4ZxlNMP/GaFopSZMUSZiNd40KrlwuwuSrUD8v58aMRWxFBrMGeuSqWWmLJW+tyqlKHoFPLDUCl1XOP+mmVyBjFtmvcNoRa7JP1NgfR+V7YzmTE8MEQDpjKoDSRle8iKj4pss808BgzoVKWxmEZ+tr8NOK3QwoKpC5cmkkHwa12aZcDBzmtbbe6fhjQBJYhKbCcls9E7iJ61Z1mzBfw6DU9mYxoQbVCZg4GJY9OTBmvIRuqKyhKkE87lq1RWXVvafQM2CTgVVSa1NcF90luizmWJU1Jvzc4e91xfu4a0BpjiTLHSaYYoVZQK75dytE6q6OhevOu5i8rKPVs4E1bYeDfDsWtrODuwrxEWQNNpfneHhNVMgxJZTyMK645ppo4evmDgvdjmkS6E7MxRsX8v4r07Ydd7FaF8FuLKt/j085ay0yutfzzH0NrLXK9Bdn/w0ZBUETyzS7nsbdZfXOJ1a1PWrv+EgvAOg3OXAPOlECgDDKZaeKtNnJppCUfMC/pgGBo29JYvpOdVpS506WcTa12adEQM+U6eVCdPLrJQN1XWYGZNIEHzkoCFm3RKbrlQA1RtuUqrYqysJZLlXlFFmWSuTzP+Z4qXoC7CVr10gXzC4jnJ1ChM3NgNZJk1bRsWhgaXxPqtbtJXZbj3DHtS04+qH9gKOBfoMV09U5QyjC6ZTnytEyqrbifo/2OqbIxWI2iEdmIgMGR1iX2xZTdFLJYkibvRZ7anP3M5rOWC4ALpDdFZzZvwNi4mU/ZHr73c4BxU0gMTumK3+DVvn62V88GMEZIyNPMthHmnuCy79ic4MK3gbWhDEVib3yYlHaT5vctm7vzR4Zd/tewyQcYKEbNveIvUA5gikiKtu0C9Pg1CCEQDFgV7rbyzuktU0FV0BqogO357Pg+WGfw2csWJEVMCD++L8VW8VcjkossCkuv2ApRajWO12C+5BhokGmxU8yFneLaylRlw7bci5NwF/oy53ybvsM8DBXYkznh4FEIzJbGtSNcbcbFqyBN/LTtCgSy5YMCwUisyTcr3maFMc7t5NZRG71sVzVqlCgSSYYMoXeOM8pnLSbAftVcQGivy60pxYe7PNmbJnAhNkAh0BNAIvaA5yWlnKgo49RQM+gCh1NGCSECTFG0NjEpL+qKwYDtalEiIQ84wp0W5ZJl2PH/FaokuMOgehlO5ghdngEufSBwbA7acVr8PwR5qBrKCRyvUfsPQip5hhfzyxZaEY+FjC1SlAsfIaFBOWCs4mKGM/QHMPLMnd2/7MojLE7CekGiEoiFbSuQl8UC10Ex5TvcgJ0hil4KC9YmBnKJv2xrajlOb2zYOxtFSgg9+ANMgcSZkoiBByiT6qcyeyek7B3w1YSdOE1nMf2dQKUuUG+Y2bJ0iqW7fWhqgXfSTlTDOggKhI4Uuw4CeA/wqIEp9bhDZgDkAt9e1jYwKn/kAEPYKhlOT1FYAe4dpxdBVwwOEL+IWUJrXo3HAt3iYa4EHHoKU8zW9jALrLhrsol9IpR/J55BPbWuigPBG79aw80dDfVn6LCKXIFokIA9QyZqvbYdG5/3Sg12p1d41LbeYzmwZlsvYySk+pWLbsobUXY0agxM6fseFrLbZaltF2UkeP6FxLQrkdwDge/CZOMp08evgEjRAvd0dftAFZxND6wJDxyOEGPinoA5O1BlHYEm8LYpMCsqcLpAB+l4NpsdaMdLc3SFJaF463RLnmfU9SnQ94gx54C25UjMOIN+iwwDtCvL6VSv3Lfy244Jj+IGhgivyNKJCZfGA7UVZ9nhpU90+us5HF6L0j0bCIAProocGKoeLmAYrMixmZaQJTbkVLtBeOPJpZuiQ9+jFaJIT3ev/CB+jg+HuXMyX2hXmAyLSOf2uGfEGjTPfm6kzGJozg3RoRi3kUug3ZeNjOrKSUJtY8RRlBSllxJxmOONIhsjXDq+hfMp/CnbVNB5MHlUrl3PLRvysOMqKkPUF1cLsARQVVwpK4RacsNSxDCN0u4wYuuNetHrzVI3EjzEz7wxjsC+kMuMB8L58OCyTqpTa4yHFwMuHfZn1zTSfVFrlJ0uUPasEXs2QR5ovNVyedmNZifiprSzZg1DttVdi2jmGfMKCV0ywH3roodBhwUlMgzVKw9VC6hmR0wgAaFwAwcLDRO6B8HzyJE2m37CflqCoStx8uDzVnbScivyTofbOO10vj+WwbUeBbez3tN6iqgaXuuhYgPrPXUqJL2q6t1tA4FdEWflXqaF213XXLIVdFLNNuGPOOpC7Uds6OgTQ+w0N3DhdHKLv3iYXYmyVqR9MxxqNyho1FDqx+KxlnzqTA+G30dhaonnFRjfXtI4bP3QaYftxzs2GM4HD2JINE/VjH4RW1YFU2tpryK8LWtWCXy0Fz43DK8gtT24MKhhsv9YAqZKXpbakLYwjgxE0klsPbCSXffPZARyXtpUf4lpI/LR1aXUN3KD48jmXFI8pImgh/aVc2C7xlfsd5l7lAtJnXhtBznt/f4Ot9HzCNjWdR0m1zTBSsFdFyxKGfdB9YuuC1Q7P5zCQ4TFmhMKbRfgNWk8yeV4C8Tz2UabPnQAY3KtZCo04qifQN5TmUUkBzdgv93a+VYPGxdNJqt9GeJOVYI7UvaV2LcNmDQrTf8MBY3ayI5yf24I1JpGaEd3G2+oRk3q6IdKDmZlbPW+sSsXyIXkpTNeaAc6YbUv0aq9WCzYjz8YGNy2dqD5VZwVNg5xSvtcPRi+2kAw0JDC/tbdihs/BGenxAbmx5eNGZhXiUKfjNox5Ba9jRIYRl0Sz6hrCm3J636YzDsYA+yG9qd9y0E1IqW3uiF7Vl0DEet+gKpazo/v3W9D0UJ6SfsGod2lta1wkSn0BXRgS/vMlnP3EbxMQTSJ5QwKticfiqZL6ykHG4bn1lQ+QAWjm+ZxFjVdAiSXuMYzMCYlylRufy6iUOywxYemKU1e2SZHtyzi8crnDIX0gilUJB6GjkIeDSCdeqa8OSoK/LwR3X0DplVFKSeDQqKVyXNec/mka9nVOhmTJ84vLLpubSikyVMl1Y0sKm/MsjS3qqi8liXp2iwo1x+xGF7JQp3spnqaVMHmXhav781Kai71uNSdGpm8a3Vj1sn5aKx2NYc20NMx6Jn/qcKSUg4pcG2Z3QS5Ive8kupaGIEwxYU1mzj6dAmzEHGUWaZvgUhtZ1YiXmwDlkxpFqInZqmPg8H2YcFn+njSzjYXClSwvBESjnYBB21wU/gOmyAtdKLOB4k96UKozYMRM6Z3cMMzOmNAH77iC6UJjelD6Yv78MO1YoNly2deatdZSi36lGIL/iupVVvaw7ZjLcIavxvlGPyv5Zf8qkWY/JqO4AelGX2akqy+asSZ+h4G00i5VZpGdn3rTGshJ1/IG6eRdtJ4VnfOKHQQhhryKnWmWrTRM3ltqLYrLZ5Q/5ua36SAq5oS5W5T8uJaZzs0L4GALix4BLw+ZkG91gdV4BoVVUL76Gpw52oLcCMjusdkOJ3g+sHN6yR1ijIPSPJYX//v1/HXvvf1r1+/+PqNZ2Ha9NXwpm4yvzo7uTsHR9ABI63W8dAcPBigKCs/TN1tGUeWQylKRmVUNthtpXdQbNpKhcmFAX/RCVLg/vPuQM/ltnSP0D8dGwUWGR0GLUUMxZcWn2i57dyC70QbFbLlwCJr2fuQfHSzZIOrM3XtU02Owjje4Cv2GiOAJEYiNp/N2Ndyfx36BX9Vgo63wmwYVPi8EPr0LjwaKA/T3NinhrRxeqHOSEBPxz69qM9IuFQuF7KdSjJBYoAS+tAErHG/wZWnN+7gEha8mrk/zvBkraaB77FNo4JUvrpiXePWXkg4rU3cZbi+ssHNWKEsXGPEv6C0jDTrzJVlWY93PHlP0niFK6WsqGBuYnK7fMw3IAFN55ZJ9OUb3logJuTNR5I0mw3fZBtBBzBhp8C1iz5Z3KiIFgz8wZTZnRy6OY4UF/nliOYDxn5tHXCcvRXU4rhYd0Pj7xen7lzzsIQWHzhtDGbgz4g+CoOkK7kARusAq9MdIK2DATozoykiN5HbJaZslEucqymD62I5MmXA4RyMhWTghBngHlqAte+wAKCkZpB0CgdggEfR0HYexE/RW2dS7RLXGlirTz65qA86ucSg7dzUFvrMLxD26yvM6uqkPxnm3T0QjKcJLs7zhT58D3gW3QWWIlnKzbBWJ5CmWxhQMN3OHZmd65TSJ7d0hjCK3GSn9vq3RB+JTENe7tAIKob8VDJ3xjhIx/MofOWr+DMRKOAJWEXJjKkhXe665GnJqCWa0PU+5XsnFvKP1d28Z5GPs2uCJdbIvpD+H4H75z/kHjirE60HMHftrtR7CzPETMsrBxZWPmdD4ICdqI9AGJi6WtR3JFYxLqDqGbPOwjgLCf+Y35XLqeCb6p3I08IapoN6a51aP5e00VmRai3siRtcPRrSuBbP12EUcXcTrayB0Y3aLkPVO6ZLy4SxLTpFwZJlzMAf6POMs/miOWlhjwdSgp7+dDZp9OPIhjH50odeXCmNKznJ1Luz+VTqY+KoWvl+IilZ52FSas4y44LPH3/qjscW2es4czfCPDB5w9uERvAwFGduaFyWuZXCNdxDFCcFt4pAh+/B2P9c4huDdJgO9YS2KVHT4fGCJol2Sd19I+JoAvSdl57uliDcShS+mxYfUmot1WfYpGiuPmibqd+pR0rMTfqcJn0+IqN28y3L1QDdPGyAw7OJYdXrg2KDStAJUpE+QxZ9H7IFQEbIBaL2MbG0jc01IJ3QoTqYkIGbpJT9vwCLFrQP0Sivz62aYrY82RYyxR+P7JRecQOwXI0dBZFkXZ/rFio1OnneHBwi1wbkWR542EeInoqcuQXluskesSRUvaRdYfu1me9GhE7rN5YcI67JUD3pQigQcq1cF7kcPZPODA80EbYCwyEyxIZH2tEJcmC0wrWFzflg/+C1bFafhSl71kS8RA/Kx7YhLlBn2sqEb5OXrAuUStSFroHxZMpunMmI7kVZBhYJ+Bk/gBSje/QTWrXJY2j1ZDfek12vJ7sDPdHvEFlUvtiYpH8mHVegb/LDBapCkIAhJ6e1WCBtqIiBPM+W8VUe5g3JGLBEok8F1KZzxDXZY9KM+2ViFPUhZnbLHjhA+4D/ssoi0d9w3Q0ByLBDvYTU9b/y9TDXBbcdxQk1B3xQEw1QojvOeleReT6gW1SrGLzIvi+hSmH2aDaVK3RGgFFHFlUYzggc1uHCzkKHjgfWcT+cGM3gxCntCtpusJXmEwVQViTKVOk+c+FDSvH6FgNaZjZv0DtcE3eqK+auGbu7potHcyth8/IVhXk5nQFIp6Fv0KIBwS0S41QklkW4WWnPOlbIV2AUY5UEd0T8/fk50bVf07VByXVv7zZR/67JGp0wibOplCfTTuhuJU9FDYQrj+w1N7x6q6hvgq1MNd7EogxtawaotLZdReMr8fUiqH06ZQMnccIQri3n03c0SSUZpO385rCnNQnzuG6CiSpp9xTRPiOH3aCWaqRznCgUc1r90Iu+lO1DMmep3GsdbedeyJfSsJnWp3Z5ZBgt6RAnu9R+3HzEha1rKYh1PYIJtZoSap7qEqvoLmBxt5qu0qaMmYTfj/oPQlQH3y/lurf8bYPl+YUlr2gXbVbhA8oTazDZUAviVBFMmEkiQV7CICLu5hVJgfkmvFiHoV5LVZssl7i9w1j7wSz9PkUo/OA6AUJG+O51HsIMItM4Q2uEPz7CIFKccRB70hUDyWEQ03sivqqsj1vumECYZk+Nv88G4M/J5DnDAwJahyvWK1d+mC9l9TBbvs+mRHZLiacrsV9eaSkifzBgGez0A7IQltJQ6JghYOiHxdaTUtWGmh3qH3RRdkqsVGVfrjScuDvMirLNLkMaLHUL0x1i4Z0z4lfthoDunJEjQZMAA5aKrMgYIcMepO2Hj45m+HBHa56dt3eECIKCalxcOtD7EvfAjPry3CPjBb5AQdatGhAQFfBW2cbYVq/IaLcurFoISFogqa9b0e9GInTNUeOIzF6RFlN/CZ42mOPCQlqSnU2C/ils9bmydxAB3JKnzG5HjundKgFh/Zn8mWBQc9eSFHdg9s86QUrOx0Bi16iTrl0Ccpouu4eVdIxTdQC9tHOHzqA3lWVwYC0u2NGOEWzUW0PHMOfBbskXQ8v3Qkl37U0LP3q5FoaYi2Ls6JTOkgWWpGf/hWgK17Eot6nfbCMyZ87Yvh6jdAcRj8Idbqz2QoH6DRT372H2DL5tKEEEPm3ePD/3Tk6fnT1+e3oypfNYUOxHYMVFyzkR3vt+cga5QernEYCC8fgYXAQna7o/7wgUO2m0+Obtq9enJzR1Lh6ZiZ21bcuNs/voUJFPhN/KwZJmK/6owp/DJ0+hjz80ol7p94pZ+nwX4HkjDQZksi3etRaRcwGm+1ohEnlI5SH+GpZ0Jk2E5wxgsELHdo5+ZnyHsXV0xV3AKJ5bgwu1KlxD9V88PzsFlLw8xd9i+dBsb1Vuz4JdNIFlWgyxFPgFbXnn9dqGLDRs/bQ9IxePgmnAkGVEUMjx1uXGIEHnb/b1CegE7pEJDhhtXUUET5YYBUQRq1zvz0VI99rjowLygKJRKDHP5O9EKQcEweCZNA0Y/WIUNTJc4OsVN4/jUUU+YeoBduhUFsAR/XL2hOnXd4Z3LVaKml08xsZCIHi8UZ7Kc23h/d2BgbHiBqsbAtbp3LMnf7sDnCwU7zlh+wckg3O6lb8aVQHC7wJBT/wPOPEvMKhck8OdxxGlPoJ49BBA/C5TnUUh0YKJ0WevTpw7A1MyAvwBslgfq18EaljRcozFDeWgDnDW7/iixVeiWINLP9yRQQo+NuDUvCRpeAyKinISWWHMihgBx/FaB2mxJ9RpHFJT7jC8TQj0de/hjCaqB65+OwJE9lhj9HcZ4tCsqJD5sUkBVoaLPoDjcwMMRkBhwLQy8l5lhipu7bFkjEfnvtbprpSi2s1hXcd4wtWFSop6cnb68uQUf9HjiFTN0VHA12Wa748o56DK6CHFlmd0ed5f1xoKJeqQDLJqtidD7uhINhDi7BJ64AYwJONUPajwEipbCn9wJ1Fo0UEBtryRYCVO1Qt54wyBq4MyEvf6CSWyysr1vQSs01blS3U3CLqeMxziXC9+y0dqAVyi76gR0dTbsWQ+bB3n11LzTHVVtL5pQyaL/ceyVpX1YW11ZF833kT+aaMxFWr0LjkUqqHOq2btoLEE8FDJzm9zZLk8GlLti3XPyb4HYsMB+DDBS/Pl8/NTeg5GpPn8zduTV7+9vW3hFMMsKt97VQW0l3je9A0t3dYvWaDFhh1Goy3LXdmbXjqb9ZgOO2BBzmNB0U6qo+rKDBxpp1nP8HSlAUMvJoMXXEl0Ou28BZHZf/rfOTBzCG3QXY7r4KziY3m8rMwMXNYSe9jVa2rhT7DZlFzl3gOrmHaWuHgceWbPHUyvuk9n1A0I1gE3EoREGSYdL4UmX58NQcjJ1eEH/XC8kkHyqONxq8BE97i616WGflZLNWS3kw6aRZQ9v2UdpQtp5DjRHjFcoJByunPX76E6PsS+EvupzlZrHyUSF2PxEfA32M9LSdnYEMDQuBg/OVbxAR1nMb6zziCcbgod8Aolzhlsn27KG0yhBF/bkuKeiKEVuc3w13JQ+OEuGpRNs5E8YAnt4ujebDZb9Pc6ydeTycQ7ffnL85enekLk7+SBvPIE/ZCZ3f2VMvyBOh4xVU3/YJ+G0ktfb8CfNvDqEILqjAb2/zAsewE=", "blender_rig.py": "...", "app.py": "eNq9fV1vG8mW2Lt+RW0bs2zaZIukJFuiTO1qbI9tZDyjyLr3BuEQRLG7SLbV7G73ByWZYJCXvUCAJBfIywUWG+wGCJKH5GGDPCRA3mb/yfyB7E/Yc05V9ReblDR3Zj0YqT+qT506db7rVMkwjPd+IqIp/8JuxISZbyPuuEGTOcJjIk5SuOnDjZ9EAT3zg0RMguCaBcwOFgEL+cz1+V6YTjzX5vDMx/+TiMfC54wzuFqKGD5lqc+S1AcQcP3KC1Jn6vFIWIZh7LmLMIgSNvOCib7+FAe+vg5ifRVx3wkW2Z3QV7GwI5FkzeJ0EkaBLeLsSTKPBAzMn2UP3IXYyzqmQTMew9Xe3jQKFsxi6h2MZurO1EPhw2iFfjUTyVg+2ds7/+79h49swIbGK74AikRGixmvgigSdPWRe6nD9WUir77mrievLkUYxIEx2ru4fPPxzRUCWu0x+PcECL6YRKLPzJDHQdxigZ1EQrSYzYHGLZaI2ySNeNNi3Q6jJjAzC+HD7wFbBDBlAbtKo0lA4Ay6NADaUYsdHB+24Cv812K9zuFxsyUbvRYJ9zzuUMPjFjvq9rCBbHjYOXmuG17y0JWtegCvd/QcX2+AO4fxYpvDjurzeLPNB37rLqjVUUd12D0qdbjee/Ovzj9cfPvm4/j1+0sYnLGPjAacuf8u9e9S7h+8bvf2eRwDI+yLW74IPTEGoDMRG3t7e46YsvHU9URsOs0+9RpEjohoziShxtyH9k5gzbwJzop6CnJReRK5M2s6uc2fWBNP+E6hhZwW90sGjTrM/ulmIGE8ce3ACiafCl9njxeJV3gMbBeJ2A38A8eK8U0FaOiKLzzWuIYCmEHf8Mh2YUr1LQxg7PrTwEIxA7bDz6fIMUP4JOTJ3PoUuL7ptNi0yaZBxKbM9RW93CnTjcStGyexuflNMwP5DGBOcxAxyI1wTJR0C39sfGs8NZrNZnFg0N8U1Q5+DgBBA9DdFITRiW/cZG6ahuUFM6OluwVVkEbYWE97nC4WPLozcchq7jkMligAQmwa/FMaJ8AnLbZay849kGoiSIbK1Hj6dJV90iD5a7RY4y8azX5rLeXx6VP24//RMslWXDaF+0ZzzcJbfGkUAIr4cyo8kQSsABjmBlubhUfzFOcSOuvAC3nTYjS/DBRRCWThq1B91GgDvCb2XXgZi1kKujQeJwEIuxrHmsUFppoa3wTRgvfZqrFIvcRdwmTzBs5HDqbwosmEFwvW6MJsJ0FjXULLrOt6iuCpV8LuSpHt/Xm/PKRG7Fa6VRQeu3m3PnZZ241qnHdUBH4JXCSifmkC6JEmXIkgP/3hf7ErV4D6j/uM0GfbB5bTtDQYZpISblaHhA+z0cBYioy0Y1RZL9WRfe3ROGo/nnjZGOs/XpmNpesIYLAGe8biJDJroMgWjWYTmjQAhilAwOVLFnrcD4jaMKbSQDc+z4ZslGSNGPMhLPv0qZojKf3FzgzS6DaoTBBuJfmZdFs8DIEMpnGeNwLOg8GgipQaCSENS1BGSjuVuvECZ0wKYHsn337/Gp7MRBRsdFJS40D7a2TkZaZUmgbpz+sWW6IOlCjlPY4sNxGL2GzWIbbg4BOMI+G5Yim2I3cpG/QZtkcHzUcOBlvJbPSb0PWL4anHNUsaWV/GUkRgq0Q8jl1/jDoHugsBM5gpQ+O7pd+pcYE6qo9eCPuKyR4W6Huw2J2lgnF0QrWONMrmrqpcho0dmDRGa6Zfs+xpUw0DyXujcVWMA0otQIMwHDW3Yv/TX//d//+/f2Crm7UCpCyP8YP/g68mlz5pgiW6fP+W/DrjPE3QZzI4/gYmeJcuQFbwyZyuQCzIeYxcTqSGF3bKnSgNhUMffARMM7Lgax8c2xRArJXBW7riRkTjebLwTGfwHfBti82FO5sng+foheF79fjGjcTgGw4S2GLg9oWcnqshg2/+RsYA7OA1M9GJFuAygMT66M6704gvBIh8BKJc4BHgyxBgUbwADguYCHSD0NFXPIP22wF3gi7ucSYMFYVYOBrwDfLJUMSGeXjpuEtQUneeGDTkOPsr+Xsd3p46bgza6K4/9cTtKQcZ9NskMX1bYOhzuslXBroD7vSurTxM3RIhtB2gmJ2AMujbgZcu/NMZD/vdDnQ04fb1LApASfWfdHvdg+5xHewJuVFtDDnSuN89hA8BUBD1n5zwE4fz0yl02p5CFOHd9eM78EsW7dRtnNWAKo6bvordL6J/+Dy8bZz949/+9//wch8anGGrs4/pBEMwTsaZ3bEw9WLO6mBONESFlTgWJ0I0zt4KX0Q8Qk7AKO/9OeiwN5oLX+5PzlRv9FMLFkZNaeR57gRmNYqz0OlzCnEkNfkMMjE1kgHIcGJiVGbhD7PJnlJs0lxnXINcm8/+Z3Qtp8afLwcrfJE3Q46uNDP+nNi8m7UhTt9oJPkfdTOhZ9K9HEgc2YTnvgwVxxD47GMwMVhV+bVR5FewT3+x+qxwK/CrlpzIHjRW8HPd0ES/cZ1kDuzU+ep0k5Ml6/Q7FbaqY6lNXmyANvWCG+CU1PNi0OvCP2W3UTsG9F3utSFSt69BkTQqwOVn+UcwwxL9CkeWxRDtRJuErR8h+qfggEOk3E6CsH+I/PmSM1An02z8EBOD4h00wDfh/nWjwppGmSePJ8eT6fQ05/nuAcI8n0RuhMop5D6FsJijgDAQdNFPv//jy32ec6eKDII0CVPQOjBxqPtboMyEDg9DmHGPLyYOCA0EpjWR0YNCoTqrhUGMbgqjxiHUfcrOKOyWDhLpZQIG2hZQw/vMehWDKwDkATKgpyDSLehLDKQgbuRRIsMmgxywsdEkdIyZGwUHzztGFmvR49CcFkDkvW9iS42+INEc9G6sL24o55CCbnhsDr+EoxqafQmVCwimFr7MgvSS0JRMWhNNGHQa6jEo3K1FeGjAS4LQqsR9am731NSHUTAD4xRLkMDQdost4lmLJWAgHaBtZhWjKIiUtQSmjkBDgBFGt+iuaCtDO8GMC7814fsFEKWLyQtUaiSF1IPUaU3tqYF5HEhASBV1ERP9EbAkSlEltgE5+WWC8asJAPaZAmx24TVeN4kv6ekZ61idgyxkdjKgGe+APGEq5Yk4Ojw+dAzSjThe2cw0nnSnJ87zCb3IPzeeHE2OJtOO0vLJLQ4d/aH//O/Zir5fV0DtlRTFT3/zV+yDzE0hpwYosivh9a3OFMMlYAIARgErEVwBw0vVv9EsIVSGvoJpXMuva0BOMf3ms3+zAhpmb6VJ2CQ+dNTc2+5rFJVsr9Pr9SZaRXch2I8DD0TlSY/3nIPnpxUd3QMdXdWfIXcwRUnOBCMtXusKnFYM8y41XPJ9ql4NaH5btCciuUFboBT0JEiSYNEnH2IDMnzhn61gvtcv9+k69xZyXdzrZC7NCn6tG2crEI71V7mPsAtjZfU2HaoX3eOuUyEjYHkagF8/BQvVn7uOI/zd9JDmVeJzmnUFxrbQFSF9Cszmxy75efQRsw7jjRlbNRoZIyKn56JT4I02JSH7GAjwqE0OBFDfPOk4YtZSktR6cjJ5ASZN3zcLCGmqdr5ihCpFo4QYDxdL1rWOYiaBow1zfXBuTxtA9ZI/VramL4kcZ395Le7IlMcMQa3QXVsVOg4DRYEO66yToPZVmxDrrJEjCGjFxM7IbQRvCoD7SYshjBbzxBSuyTMAvTwXQv8a0/yifl3AxI4noJCB7cM88V02p1PgYJkFl+q9RUsEY3k5S12H+7agjnQEBDdBAviQCRFBBZ40JxHCjCmDjMBB8hM+8YTOtiNAGTCPAz+7Bq3Pr4XKakjjoUwDqWMYPOoX0i2okGm0+kkhqOEucM8sst4gL5kG+e0QBsuUvqf8d4izphHGJOCWkFc/Dz5R3E6puLhp5fF5Ro+8D7xDy0OLKRb+QjOFGfmnTw/QjHSVffJn0Cxf5TCbtcPp5+R7gkovqGCT9Rui6wEQAZoFaslNxkQEc3P+c6dJ9jSQ38rQnB6pAcYJhdRTiILB1GGqHOPhCxnh+zJchtmEh9KAGyCi6mYtbVcHABRs7B491Q4CMC1Ma8EBipMhdDZSll7dgcPhBTyBxgVvD98tZMsc5k0QXZsFcEl0V3au8CvEd6TIFKX+zxac+nRJqbOiYGlZojhM3Da1QGVSdC88zHDQFJG4wY/mNqEryNn9WO6Qw3s/VrI5kBOk7prS3dISLBU2ss+94ChJO0AKEe2a7OUAdHIrcyYHeKFGFw8mQeCZBaXQvL+DwB+DyMzEQAUeoF6cPkyUFYsEGIinHggAtVC5KZ2MMrFlceVE3NoiTNgb+gWKGlc3RZnZ9FIouIoCuav0MntqhREMeAzwzOYGr6I4Ia9ihlgo4QFbOcgXXK0rujJVfIcSAJhy4FJ/cBWlKqSBaIAiE9XFtRAYSJigB9PQQevRbGFG4JDevrk6vziXuTRKFuKyoWGDmwMkQZAg5m8wdGfd/QN0+T5QKGhSVr5pbMyCoXOaCAauAYpy/RWYngRzpVOfGwCKybh6EAcSRJYvYXeslJqWyggEzodhSR//Zg5RDBLGjccQSC+FWdJDyAJEeKkUizxRjPg84ZvyHUaT2EGZB56w71Kx5CojwvJVf2jLuJ2k0PUXjuqXs0UA08wjj5fZQCJd6Kj02gf3AiacUMW3w3Z3VI4nW+wG9BqfUDgk55bGJL+s0BMfVmLqO1d4DjPLIaJKci5VhvNGZzepH2SmZ5vcdcCe1QmoSSt/9OH66VOKWLjFPlKCGugx4Z/Q0vz0b/8bYFkEWSvs5bAzMyDKWmD0WRJiGHO/ZrAkH5URlPp+UD+5GVJxfMc6ykx8Jt15/39i3xg2Y3AtA+oMfgEPGWXJgIsYRhlDqSLGMkIe1IfEyylMOUpEJAVCWphpinYCcwKlx7i+Dk/RZ19J0QtS9CSyfFA0NBw3QqzhCtMHeEl6H3MIGZGW08ynA1BL5aBgOFBca5F0wx6AbGXidHOyUM6BNGKrliJPdMiMxThixp3glHngE6p1Olow8Vl1CW7vZyCg8xxTQ8Gm3OwyHjamjZHKYay/MjSeUlFh4LyNADRhsQeMY3atTnOvKrlVQXy+iWdJCOqRlikZxSZl6dPj+ce//Y//k+0aFKUN1DAai3iGq4y4CAx4NX8un9SP8hmCGnb7hyMcLF4fjrbrj4dMWnn8OldVHDIlW2IqsJDB2YQn9txU6TLy2pR/ljk0gNCFujabeTxDFRj42T2Ry5JHLjgeGLTExs6o4ouLvqgX4MvhCO1Yltx0WzK/Kfx0oWJJ7LrAYBpd02X7ZIlkgxauSdkDLGSgGgugdHe9v8ob6HU7GZ2Q4zKlBdTY9UFHgNdrTlFpRipDObXQ/mx336OC247wiKiDOr+Y6Dygn2VrhnTQTh3wFGZRR+UWQKPCEuTf/BXL1iAmVHUHehHvmmv20+//U83LaNgAqI1RafQPcBfL/WKqb0u/fbYSlRVRObulddFgprlwcTemCqfYzNcd33JPAOtgFCmrn2IIRHjk4LItZoNhdmDWxRKaBJWYuJkvNeIyH/FTxkvO/VVIsuLP+v43V1hppkqSULyWIoqVii5l1YurmG4s8+5l0mHGzfXTnHPcBfK5DzJqmrtKrtBgwVDHcqhW6M8wopXPKC6kR7VrDA9bnYCB5QsLunxgMasECkjHLNKA1y22MfVOc9jvHo0sCL09iBxMY4yYsqyaSzECgRr2n3dGavIDACqn3xTLpI8K5CO4yHbymidckRFIStPo/JJTuLeFVnoGKUvD/TvzfiJm83VvfJf5lA+bV1XQht4z0mAIFLJc3xG3o70a3nv0Croul52nievlsQXdWnYQ3pVh6RtAhaZ8TEs143Fzow+MRSWGYBIxUFPJlqK+VK+wAtGC6NwxkRM2kS+XKmZLJxV1VTAFPI6LDLe5xLel7E/FUWC34b7bOzjMrOQczWZ0p7XTF4q7Hsl/tBxWz4NaRYJYHGViAdYsSCNbZDqxTDs5cWFcmjgdzweRPc8eLheYPaOG1tKNMKAbLyD6xvFkjVAZ0WeWnTrcgvdjNNBIKZOyJYWXGI4usXJ54qEDLheKOlibmBuTOgNfXkZZKRKhgY7ciTCFFTbXaB8uzz+AAVkurDQWDphzTEp2+lZ3uoabzYIFAAVtZbFY1hiXdd5+jVVUCysUEZZvrL+iarS3F79hacwd3q+HZc6G3RFEFbNhZ9Ss9r6il9VeSqVzOais8iVGp9zm9lzAuND3EBbdZfa31vbebhS6UI7lFply/O79xyvKfyRGn7wlYxam+nIZ8YW+LlzaeQvQwSFdr7MKWUqyaV7bZC9kkCUyCaZlwhtMUlidTb5Egc6L78kNMnxwclzejhcua7c/pyK6awOyAwTtfqF1DAvuW5InadL1NU1qy9ilVGko6BNClEdgwuBGRJYT8RvojTI+ycCOly0/mINbLCK4SH03iY2dubh4LjxPudU2DxG60iPqIaaM9CUEN/BucNC04sSBqwwwrtEOZebxVtqIWyqjhnaRG5pNmQQ3wSsymsBu+rZlNPMMCbrk0EfYklTn0I6K9iFi4ZIZMRHNhz18DOqFDw9G+PMwB4H8MJZrtMDg6qtup3dIbL2/wo+zJ5KnjXuUbAGkAa4ySpX85F5ts+ReTNbMTkxkAxwfAMNEDf3K5BUID68VKLgcq+eYYMBJHyRhXrx3PW5J2iJ4XQ+Zo0vyMrwej7QPc5urJCxSy94P++2TTqmIHAG2svHC1QJnQjuuAbBSoFJOBd814lPcDANiv3SXATNPOixuWuwy+BQwO8VsEePeDDfPxLJKznMXgDSIz5zf4Zo8z1zYknjVI4PxqJbfvQf48ro8vrQwWipBa5x9kCMD94LhEjLcgbYnv764nrgllPsUTGL5nCeos3yYZAMfoqE1uJ0AUXBJBtdrwPnD8eKdqrz/9v2HIoOcvFAcctKTLHLSkaxxcqQY4fhArQVBwCBr9k0DGVJpxVZergCXmLYA7+u3YGUMrSuxhVx+1S32qt6aaq5bq6qIHN4r2Z+t+hMd3jnobIV2hfpX62FsfzidHDgnePnj37/S4v+7FnsHo3mOdSQnUtmihwEcYM8hnCrHyRguJeVQR+ZPUTSAwi2WkttMNMp5ASPejPtzNb6kfTFzcoHnmLFlZwOcmFIru1o5Qp+RPwAdFgMKfFGOKAhdLY2U060mreeYr+6Wvwpp+dDQddOgzX4HehBj/uMTUmat1Tuw3e/gIVbg3JKmbJJukzZ8py1RqYbbcqoBhn/S6Y8qOWeaiDwafhkG3h2uwbMQEEviQWMFuK4bmCPxBg0fIiys8IuCaxC1FZa77cBkasiWbao7gPY966hM3q71fN3YPyvEfcQRBXywOEOLti4rOQhv2UlWRViqRClUP2ytsl411FwfHDTK6DRUjUtjnZV9rDcLYbYDtreBa5ytkDfW7OXkbEV8SAZqla5lLQmVoCgqoBbBcALUjPTktWbJPFPohBqdsU7OViVGJLfU42ty2PBbBfrXJO4mYQnJLYTVTTcaqmKgdV2dMPRxResU4IoiJTMaNZQuljuLkKbozBJYbEe02KQ0GKsx8Tr6Ey/p6rY7aHQa7LYH3Po74O47uF+hGHasLt72SrdaENRYjo70o7bD4zmPIn43aByyQ2Rx6hFdN1wOrClOltQ4++mv/459pHJ5J6hur6ApxoSrgWORpWNSXf5QWmhh5ZlVPUhVDz388Q/sKgDLLbdGaJL8idVh5QmVWFWnvlp0VMNimguPsXCspwrHqCiW/dplY1QTfxPxsI8/qBieasdWOGnr2gqyLT4H2mQI/JR3QxV88kk1FAQVsL9RibU1CMSMfngjb3+ny9bqy9Di5YyWDb8OboGfWYcBM7PVO2BZ3HgpoqU4j0NhJ5cYtuRqfUvh9otOTY2frLJDErEOEEnL0nrVaEhuJcuCwgjIPHyOFNkrswEdaLCkw5rrSo3+3t6rj7iQjt7mJMBVA0sWuNNkc0AsYqvNKjz2ZzJC5BBW79V8ASFJW9HjqIPFjHn7U1Ziw/Xek7kTsXmXrQoFhMjBTFGqo5qEbFXeJqEb9JCUDJvFriMq6E66Tu/wlG2tymQVYUIasmIdJoLFdetfAy6ia028AOt6VC/yttyZ1B0lGiroiiCyz9KkPJlhSW0Byj3Vhy/sQ2c6bdZN1HQ6LT7e085SVnr/ojLBm/qp8vq2DVreCW6wrJD1DuG9wuPoCDCP3FkMIZGFDMxWirHXTD+XhQKrcie9MpOt96ZBkCAnaiFBUS0TaIGle/iTPYXfsm6HfoGZWgWg4tzkrt/d+EbipZsqJLf2gnKlVtpS13PGqavDxC2h04VaQApVVYxYUC4tsugytr7msTDDyMWy9vE8FQPD9R13hnVuvkiTCMJtehp7uMDd2u3u+slgmIN+GwQzT3wDT015JgRGNkZmPtCextyP2xhCTI1RE0uS8mVZ1B7jnN/G5PgaSl/At3Xvxw6PrguNcmAoBDXQpCQgtLoGGpxuVQVHHDMmrsZWJDQ5rMLbDJBqUh4kpn4yKFKL6fHlrzQI/b6CCzFxGZLUahk+iZt4or6vQgY9TJNdRK9tsJ3qaZJg/ZkATe4gf9VMgN2bHhCS1ba7Mb0PdEbxB8DfSlp7Luxr0C6KujXY6ynf3XLr9Fc+q2eErY0qeEt5nwiqYC0cuYF+4MWQEhPj2cSgyPvgWDmo8vgOWXXhJnNUDF8ju8Qm8cvAOL+QVSK4Sw8XvbFKAMV7QD8BuzgegMmvVQyf4oFhNgdnKyewIfr1EwuZ2rI9HsffujFESY5jNnAUVBGBWSVHLAp7ezVKl8GNKTyxGLvOwADbbVRWQ6HJB4ACFgBj+CesivQP/sbqwEVpsbrU0WfMMardrcWq67zCrf3If7h9DLBxv3xOXRE5QQmc7vsVbf40Yxuc9sEBbfkZy4D9oIdFTnr8aOWrBCjCueKTUq5ySxvTUPsvjS1ti8XS8Ml73HNgEg8OZPlDtj6OPHEXArOErme0HrxkyWgbAy3VDozLt1+fG9mm4t5Bp7kVjh7EuW2DjnUD3zQ+/MN/iVVxODMFeOSew/FAi0hVomD9eBPA44pg7Zxu6wMZ4p6mJHUg8DVkeiMxKdNnx7C7vR3DzqomxLRuSi5osMhj1i/bH5WGb+8QjNwv2KHcxLDZ2bvSBoBKf48CPtaH9EAXr6MgJKVxLxGGcpNAi6rlqXCeyII+jHwhC+jpDbZ5iBioL/GbHE7WAT6imyp8Y3Q/7CX30GnbhrMi6vdACV8dkoASYjxK5vLSY/PSnWXSValm2cJRkv6XGOeZWERmYkkTVjFJxM/lSQbxPLiRZk/tnXy4bmG5wpR+/uMGd+7ROSmR4N4jBiYZUY7tI0RvIjI7Vq/Fjlqsa72QG48GHavTzaZgIZIoK2R7qLLDIzhw2vQpHI/Qa4VNCoTlK+VcmHLpUWGlTu+g4VNh1JbTOu7ppUIIGHUPt9R0rOcZKXJKYITgg23jzuOo8QpQxI8eQQTKvSQbLKiOSsu4EF2pDL0LoAUVdxiPYkJVljLI/bG+PD2NjjyQJ5McdXvsLjua5/BfMDMCLxX8JaxNa1rMeGSHLD+YjervF8AtER2sZjF5LFqfHamz3Vq6gQeGPLhnWgHFit4cGrjIi6oKz17D33i4GpasShJihYEi6xDJORoejJqPJaCaAX2akxneNu9B9JHWO9vhtEsm/mXqJjzCINe5j06KVrjIvwuiHpA8/OJxXP9bWX9rBiFu+OAeuIvoAtEsPsrRKewmquCq9kcoZPVhHarph4vDRwtCWSC+w7VpW8jt05JPZTW6i6kCuY7A7ui8R8GSwEEe5XF5k8tOBAjTCr/SoYqZlckPVlSDVOc2/cP/9mkRnSDcJxN6D9mumabZYnimAMMDEbZDnCRKL39N4Wrmo9efkWIUooKZ8NFgul9gXN5sRxckw7uLzYrnIlKpmSwdbA77x6OtcGlD/W5ew1JueYxibOL+WsomxIMhuSmjjFpvPglos9UqlmKYD26sdjHEu+KY2XiDsud6K1KU0S1e7KDbjHsSwFtQ2iK6057pVWBzlvqB3NHE8cAQ2lJR2Pn0GEmRRwDFg17m1h5h9EdHpsjiTnGjBHsr0PlYbioAZL+B+IdK08d2kPoJOBx4vh7MQM70//r9RaEU+iFEP19yHw+i3EXxR6hg2vlY8hUOaJ29xcqmozNSbkP3Z1qQCzrN1Pzp3/3XLm7Slafm3SPfci9rCbsuFo7QsVdl/Loav4Ne7mLT5zv6oD2yZfD6lFL5u9xHT/eBZ5k+hgwKn1d4whvaCnQDHuJ3PmT+5O5vaPxdupjgCHoHhxkFPooFHhRq4BYQYbt42ujgnjgt22G+0yKde4InQeTuckb1VuMygfEoWHDK0RdVbuhRBvUbaEIOJeN6c/wO+HITcgl6u3sMcOmHho8FRtqMuqjNY5ctXNQWSYRW2/zx75u/zFRku6Crftoh4vMCNUm3Q/gdHnYyLw2zg/rDh0SXO/jrUsSgwKQJfYj5VHuxSxQ8VCf/AlrqtaG5Pmcr5fYzebbm45ynbwM856C4k+hRLtNkrA8gukfB0mNMVICNMygpYuRW7pvCDqbtPT04VM4yTto3uA/yn+Qe3Qt9w9xeUAUv8L4XJOLez+WurWJa974vHmvxdgn1Jva/FRFLY0p5YgXBPpPlgA9wG6Icsy1j+RnpZBHbAmbGfUAi+UUxc4wrwnVGe6HTAe+uPnxrlotQC98vgrqJp3XL/OvyxkKsyHwTY0E11qhO1QJCvt22CB6/rIGPHk8BfnFTds1ZaLgRN40rBDdqwO4IqujIEdDRFIRwUDzsx/9Rjn9+/H8P0hgP1NxLOidc4kw4mJsCl6fGj3dYz+UY4osaQG916LEbDLDqhm5TEF4Dz/Fohvl1EB8KRhIsXnrGeGTPseaLxaB5pC59kAWplVF14oM+3oSHoXc3lvkaWdVZOd0EJgH0LLAUYKw9JWxXjlP0fgfdGtMSSaErGDdVvyJvtdR0tCQxWzlRWoq55Dkmo71yOsmy59yfCbOIcovp38P6o1EScVvYiwlqx7I917429RlHdZQc/irnHv06px/9smcg1bomNHfP2BC1xwgrk307jcArsO/GVASP3g2eESIiciOK77O4ea9urYDiLbVp2ZBHgaHugrHPQP/I1Y0C3xa3pqvlhSu8LG9Qp7eg+2COi2q2pU4TohqSbV2DZU9FNfajyFZxjTzipc/Mwv7XVr7ZrKl7Gc4wYURRYoH74KEV00ZJM99CqT9BKhcYtdAp7fgGmBMtJOS0tLSHMaJX8HlLmvUtU1DY0VWAne1Z02iQNS1utVgEcqvfzx86wSDSmuo8nPF9PLTgt2My/XrBSekXBIWnQT7eopesu4fRdbQ3/vjm8rdvLj/K3Y76702A5I6xGsjEH/kOkVeuiCCgALsFQ8G8A2e28BxOwALG/QSU+h2bpDFlK1iYiijBwxIn6i+mFHdvBva1SPKtMHLbsOtjxRemeG/EpOhEYJSmcbXCIDSva3YAO8WjEPuVc4HuNk2jAxwQxKLiKm3fP7SxWTOpYmUkqe8LXM3JkcN9iPWIJVYiInR/E40DkiJEUkSk5pFSLUb0egaxerNStCBpaMlfJpU2xA8YdmxNXNwRbXQs+g+j5honR/FbWEec7z/ScQmbsEtbxjUEwF+xlsdT356P5TTT8AYvjp/rgdFABxv8l0kQvMsL0nK5kkBNqnSNxmiYB4WhqcfUl6Sn7F3tg8PKbbFt4Q9TYcJPxvJYqDHWrGSfgfqUR8LIB3p5uFuXMaHEmnDGVCU7GFb32RZToUphaK4aapHA46pI9DdVAcmsnf1tIicr1rsVpT84U2hi7NiIDZ8Vj2wq74sEFTUTtCLd/kw/v8f88O2ORVNjniRh3N/fnwHLphPLDhYFTIpI7YM5FjwW8T6WAcbJPrrVqHj3ty9OGYXv20Co9LbNF87zQ7LCmNKp7JqHsdpz0N04yhbrBC+OjkraFR6XWRX0EP4Noxg0s0NH6KbAUAOD4w4ozb/d8gkjunl5dzc+waS4/JtPVhJcg/lLIy/mU2GeNP85BaDI84ynyXxg4qhaGZ41q2fYbLwANsBj3uoKth4uK3+SeEjDSG69DIx7R4uJURUatB4liQFCFw+fLulFKSdlCSqe0WKXtwdf0I7/ITFQru+NdtsP2vjHEeRJOPIRdPuQQGVKQgIy0u29oHnr9lc4VeuHJMrk9t1BEcP3F2/QEQfXPCo+/3j1Goha2AZccBkr50lmdkYe51Y+uoq9ZM87lVNW5M4a7EntJ7aQDbzyRnpKA9Dp0eAEYkhnRpl6GPL2l077pD169oMF05PPxg+oMjCv4srzqqpxNkjdYtMQyQlfWFghGZo1YSj4JPy6cJZY/cF/2uUb0mlZaJ7HaJ4L4xyVzwUsHQhYZkrFKyNFp4dseqVDDE0j/8Nzla2tqHwzLwOHvHEyq4Lwm5hyI7hc5ONBJ0z+6TtKNsk/nif/MF4k69oCy7KMigtc9ZXqFdKGUiJdIxVBvarZrjlqWf/xtjRnB0KMMELNWzNFG3pDmQVoLTV/jvjePwEGq3q3", "estudio.html": "eNq1W91u20iWvvdTMEp3RCYU9Wc5tmQqmzjuJANnkonT0wMYRlIiSxJjkqUhKVmyImCeYG8W2N27RV/sAnuxwAJ7MXd7MXmTfoJ9hP1OFf/0Y3e6e7eBtMiqU6dOnf9zWD6+5wonWUy4Nk4Cv793TD+az8KRXeFxpX885sztHwc8YZozZlHME7syTYa1wwqA5XDIAm5XZh6/nogoqWiOCBMeAuzac5Ox7fKZ5/CafDG90Es85tdih/ncbhKOxEt83j+Nk6nrCa39/LiuRvaO42RBv91IiGRZqw1G3fvNVrPdPOzVahMWch/vg6bb2se774W8e7/FWm77AK/JPOne54f8iHO8BVO8HbEjlzG8Mcfp3u8MOoNhA2/iCliGR+7BYLX3cDkQ81rs3XjhqDsQkcujGkZWxBNzINzFMmDRyAu7jd6Ye6Nx0m02Gt/2Bsy5GkViGrrdGYt0otToOcIXUfoOaozeEGzpNvcncy1exAkPalPPrLHJxOc1NWCe85Hg2vevzJiFcS3mkTfsiRmPhr647o491+Xhau9+IibLiYjBRxF22SAW/jThPYyCmMm85/Nhop6ilEI8ul488dmiO/T5vDdik+4hBpnvjcKah53jrgOB8ah3U/NCl8+7nR5B1q4jgNL/VntWzEfLNTRbp5YiMXqKb90mHVT4nqupWRJQNlmLmOtNY0XbhLku8bs9UaS1JnO1nTaYJokITWuQhMvSbkkE9kxYBJLXuAwp57s3NnZ6XNoIz1oTuyiJeOEYjAamaRQD1UR4khPXYzCmhm0c3g1FwYOUKEuskXS/5bSGbbYl9LU1XbCPDXzuLgXQesmia7U72b4uH7Kpn9CCyZL4223i2XULUUfcZ4k343K4v8mUrxfBDhpdV7MCHk5zAYci5L1tJVMa1VDK1tyl+b9RBw7wHHihchbd5hHNBWxeS63toDEbFxbBpomQxFtiwsONEwx84VxhlgY1ZqpfJYd1mF66FxlywudJTZpFl8xo24YlgMsdETHJGcWnlPhD0iuieP2AdKYSn+SSXEvvUMGc+O6Yjrx2BDW0QwOzRXHAfH+5ZR3gG0u6UozwJHw+uUWPxFUupjIOGiaKa9eZPBpAMxAgKdjhk9REV9qadEsd4jFZ71BEgbJj6DT/k17DjFHyPmuORroi14u4I9GDoGkQ7vJemWMjjVFSPTqYXYNCFnpBvNxygiSZdT/X+zSNE2+4qKUxLMWcocgU6DfbXclVZWrS2rCDn3VRG0RZu+hCqNspSDmuBu4Ph0Og8sKh2CFCFU0QtKTNHxCN20pFGoGgyZWkMynu9+jcmek2rQ528fmIh+7yNs8i16da83jD9vc7m7Zftqo0EXCcX+16cuvdOM6aS8zPlp9Fc73Zbs3aoaBp9gC0WmOVIfCWmQfCojyt2HIj7RItXihZm/q4+zDqXTz1QiRr8DFr1O2garfOb8s5O3x7teewcMbiTWd7XE8ztuPYibxJolFeaVe8gPLCgE0q/WX6Ele6y0oyjjivdCvjJJnE3XrdcUPrU+xy35tFVsiTejgJ6hLo7xpW86BhNeqDqee7aswKhDv1OVZUTIWqDkmKMK7/MpR8zgLkYHH9UxzUK6sVDiGJxynqKvelxA9vELTmuXYFhoC0VdPkgOOzOLYrCPIVOUlZcCynAaAMU3NZwmozLOTzSrZAhJX+ewSTacSO6wquvwmPFAeM6r/1+A2Lc6idqJU/JmAei/hWjNh8Uek/jRzP91kZ43Edh9k6FM65c8J11WF92rSS75LOIqBU+j/981+0s+mN9tM//ndBTQkFBaldbPKp6pCVQCUrCXafhgDnUFNPRIBMnzQdqm/cwSqswg/MyuFY9jR7vHULN2IBIr3DKv3nePzyIz3fCg1bCAH5ln40PULdE7NIS5SYC7IUT+/krMOCYDdj/+df/v5ftZMvP8KVsF/MXceuNMxGpf8d5c589zkA0zyUUKfIfn2X3Qp2JKHOmCs07+bP1h1sB3CtBA2vZt2Ktd0xm6gt2/X9W0Ea5mGD1DjyBrdTN0TmXumfsLKB/TzjkRLt5vvpnPwWxHknzzMcSgBr+2U/mRshpy3pG7EQPIE347746S//tgkXJ2zEB8wd8Yom3SuIvTtLp0jdOSwKwHJQ7qwFzbQILhKB7QRkR5JSCpBtCYGSeQy5XncbWkNrUiBNMR8e5h6kOLeMetvjlIBsj6pkoPC3NCaznltcU+6IEyH8TUdM45FI1pzwUyQSNQyybScgDTGBnziBtUQlgC2k10hOYffC//IfIxHucMBSkFfcJ6P685T7PBG7YcYgDzb+D3/VTthkLToUXjr7WQuyKhriwCrAag81FmvvX747PdWGkQi0qox61V42v3wTDbzkBKKMwKhVGSiLo046WV8DRTwtYXlx9v67M2gy0tFdKHw5FdcLsPXlz989PXnzFetLcOsI3gkRnIYzLxIhbC7ZiYQX83F9Y4HCtoejxon2jR3bfVc4UzkDOUWLc8jKSUSkx0Yvhfru1dnpub2EW+9eVJXZfrjmA2vkD6pmNkA6CpNYH1SRwLvJJi5NGeKBZiJjfDqYBnMapmiejlLsxhBTwVsNrjKafv/q5NReblLTrb44e6YhSvNIaBjUdAQxR2iPtN+9PX1h7KZWLUpHNB3LTO2FcEViau9l3vUpLq2Ei7GGgzlWfffsT9T503im4Jr+feglC1P7Pow4803ttYdUS2DxnpYutwY+fAEWP6NfjvitRozbeabIi73SRgUwUgeK1sISg0+AfPPsd4BKZNQWpW0/QHsiHsNptl0rTnyAnr8/09LRL/8Vau3nmh4EJTJ84TY/dBopCWdvnmdsReG6BtX60OyUoILpIoNsdr6VNJRk3a2q5E6DMmvjKaQNXIXQu9XXbMIQLjU5hrmy9EE0/LZGpTTD1MxzufiAxMELWWQFk30A/JHGlCwdOlcONoL+t5H+lsHw/rf/lBDg4ocJpS/MmoQjALwKEINCIkROamqyusotJ5Li45Ed8mvldawf+ODF2bt0XF+ykPq+HotR9k+5SZzm0YxDIa8Rb55Nh0MUbTS1gqFl6CwUMG+9OfffUbtFf82SsYXz6aqlXMyYLcPolRedIzrpXhjy6Ie07YzHl9KuyujFNJlM4dkQ/s6p02cr0s/fvXhWDBaI4YY5JDIBwSnk05PT8+88P/Cc98VciRIZFjFs8VB2/mw6YW8v9zFUWlhYhQUnY5Q2er7UhZvyOQHljidGYcZLHD6nd93oyXGriOslEHkKvTFXbfMc0ySIeFACe/v63enrF8ASMfJ1GRHUIiBwOFDUKrZcZZGDVTvT+g13qhsmaqp9w1Jmy7MNoZbl7eDSJ9TNmfETFmBTHUlfIa16SVqErmm2Go2cdifxJaq1oKRjA3Mn7wCeMv856jySnBQBDVMP4R0if8J3jZ1POOTVtFrZxqPIK7P2BV5fcn8C5W42zH1kt/MWbzfVQ6u133IyySAM6bQ4P8LQF6JsKq95PNZL7EH9wF9wEfAkWuitBo5vmCWxp0pFJQ/z9XIzeWXIFp+ILMpoKCu05nZNms3bV/VWNoeSB6xXeNTRC0IlSE7pmAdeidCXeI0nYzD5jIQDxRrK/3DkNv7b3zetg/KpaXmhvtOwhOp51s9j/gauFjBMQ8thcbJGIsaUPVkBm5CBk6GDPfuHJv2vvC9g820jL7h728NDxrBts4wAi4BgD9FFc6YQ1dT3zQDuJn1kEkVsL1cIy8K5KlscvcMMkBWXRp+JeRuDL6FPhyYlgkpzFDbKfAkVmzFP6fY5DpYfQOVAcrxIo/TMNl2K6HKylCNhVo4Th55zB8Epegsl0Ku/ti/ie4O4LlHWR34yrFeNniJL7lDaWMIQ64bTUDJJk00C/cpYIoFVJJ/Zy7TG714UUjcvWua+2b40C4lAm8ym1TBJ/pdmVu6ni1qua7YtLOuYUABaNhi6h1hmdUzriJaqZZSs5zW/WssaBzAqQFzU9rGxXHzQeNygPYERhiT/qfV5C6BEbBt7XLTNjllr0trhsNN5/Ngk2Vogutkxh8yPsVY2BErrrH3zooFlHbVKmQ7WHch91aLVxdVlD0RLGyD3TSx+yef62UXjUpkGtX9DVH0L++yieSmHsipQ2oRlWWcXrUvDCqZ+4k38xbnDfBbpLxEkvWATa/tSDZex7ksSaHQNca318KVJ/2rth0BG5r22rHOZ2lAp5cbw48snKoR0Sd+3QV6VUBzInb2hfo/WGcsdezyyDlZ7q0LDBrA13VjCUK3pBOU/4nASefMfRISASkKkvu8cXuOKnwaTZJHbzmDLQmlvwoNyb4YgxXVh95cgRlheTF7ayF7OryhIuWoMdnYlM9EwJQCIBNgcILngzygkU4pDRPYGGJ4sgGJQDBuUAfgLRfU+5oKCfmMli00OzcA2ozQo/AzuHOyrNyH+TEMwUx8YqxXlUoje4KyYl/g8hCVwnd3YsA3u282WtGnF/ZSjjk2YRjw5kT3kUlD7oyyjwGLDjDMgmaXtAum9tFWux+Z6bC1M5ABKNMrllSfnJgHE1o3xsGkd1eVMwkJKCKyhmD3Mgh+yWwMRMEfCbHaTT1LDi9vcLw9IULZhAI41fyRhUIDozFDwjoh1bjx0TcdaFLPpyM2jHGYTXh0JSQe8E7ihpEffZbBryFlku3UkPvJ1SG8PO+pF6djbSHxSoUzJVFe5Tq6AZQuhsiiGiVA8S+yGObMbvbvU/MGDe8KaIkF/Ds9pUXcD5qWSILtQr97skT2yWILdB1DGuGCVA71LeglNy7bTk/RXTdTb3TuW1dukgLmcvkJZAPuNXpVfsKqGpZLH96/P7I/fLCW/ZVKsJ0hKxZmgeyfn2Doc6cZKw8OXH8PR1Bex9re/at8sZzuAZl/+PUIA4PHxIOp/s4TOAeg7ZAOu3sL0l3/SaHCxa/BmbTD4+IisWSdKLVnuxU8+qn1LQytVCcYfu9UqzJ/Fi9DRclHyOSqwWB8aS0ggFUpks2vmIa/kiTPWh+YS4hkLt1t9efr0ebUwZxQ8VyuHERAEmg7KqLNa7W1uFI/FtU6fM0qR22GhG9uyAXJBU5c90qih53OZyfRQi+ppiquJoQI3oFcys7HGjOg2lhJ+2BtEnF2tUmdPY4aiKJMnJRiQp+x5WmnL067SR6xqoR6j9OhpNkI/T+kcOiGETUu15pBCwnyjhJXqk5P0ntLH7SastqY6zUbjIZe4uVvPkK2+/aj0FBvAloylCmsokcSMy5EVZopcbwOgNNHbSAhXKgjBeOSK3t0BSWxlymJXhi9QuE2RdQUn2AGFjRR7TwYXoCNh2rZdpSZT1ZDBQVYXG0XKOTyri1SqKD5U6xi519HAcQ5M8Gs0Dnkcd61Ox6TbYb56a3QUs9Z2SxtdVTrEV+8INkB4srBNewmlTR+vEL7UTkUyT7LobUaqIJSOJfBCayE5nHuhRc0OwrRUKgbhMamGWx8B0qIq6BVVQY80Tbap1xxStZrqC9wf9WQkuOXzcJRAkCmm/PhPM5DXNJGfItN82Ze3L6rvOEiivtGJav3Qk4hQBuPhnPlTl6VPiXx4BkuUD29FzD+8r14qnGsUxSJKdJ2ZA8Pu63If5b7fwJItuu1nPDo6Mr49OqptzA7WZlNyC6fgkFMo72QsU55dOHLlpS2ZYDm+N3kqZygeSizZsQd23jxx4EASnlb6elX1yauUB5WtW2GGScB/OFyvfsDZNQklQuyD4k3HMcm56ArUWJNeuTkzSLMxCZxRngng8kn21C0fEUl7ilUdLY3P1NaQ2MiBwjGewwWkNnmbuZdTzmxF2jAjgsmJgN7sjpl9L4UpT8rvHmcIIJkXqkqGkbXw5AeEeZ2AKd6vAdOHAOaFsQLPA676krPlojet+/PnbER2u6tPqvK6QLVbpe9X1S1shb1sYnpSPR7kX7j7FI1PmMs06YM0W5uGKnJaNPNcQCdxLC3gNzhJaGo+0+hSFKNBVwx8ZlW7Upyb5NEm2Td3uUv1kQrZqnP7+TPi8qOqpt5MbMtK/duuMnKVK6NKltUvRHqmeo9pJta7PcDlTNn9IeKp72OpvGagZRqPRDc6ZYjqA7s/KMktEaORr4RsDiz6MAoxWzOcVcZ1mSOmLQ5yNsqLFVmjVPMwy/yQMKcaH172KGSzLF6rAEgYHjxIH+7ZNjPSZySvLn8zRVnaNnrMosZvohty9FVIg5bcyOhlZLCfO3352tPXnr7sEmw73PIHhlHOl1NrEKEs+O6R700Pq5hxHdsXl7dY6nrefI38dhqPdWHATvGSEXtt968tdYk1M8Zr49b6814W7z9/3tygSJpkaNkqTvWnUcQWGJW/ehFpjSfFc/eieL4sOBrY/UDuItXZBjvSVHKVb5frx/VmQ7MoFMz1mWcs9pztFOJwcDgYDs18OxXcS5d91UCp3Wm6fJKMf4i8hHdVGmvk0eJ6nU0qCbq2VI/4jYydLSRFlB9cU8m7yrt90lLtatqiqpa69duqmN3E3dTCLLRQAsph4WKCcm3CRjImFL07e5CqQBrE0nFhu7u9L0KRctd3k1SiZm7357vcvsRkSDsWRnk3Ykg6TXlUvhHGT2d4ICD6UKBX5QmrJkXP/yNyjDt5faGu2VzexmsiZJnLr/B3/qYTXn3NPs7P7LP+/UBFbnLqkUjuDLQeZUkZaQ6FHbqoghT4F/dRUtDE3p50rLmpWl5zpLMvH1qNx9SKyPLHza5GAvDEWuDfzSMAdzqqlVBuSyQb3YXCDSgqLpiJ7K10MgvhzENWZoIbAZvov58GA6p10iaSyX9ODHfGuLIsZKFaCm+kRoUk7pLavfX3nfIrR5F1cGOl9nGSaGOfLPjvlbKpdTqUmYf2va/It3o7Ycp0kV8uhayUrjTnu2Pjraxwx8ZbMJsb7+VxoFzGFl+r0s5o+qlMVjEFYB50LOnK38Pppra0C4Yuvaq/KWqVP/aUKmml33fWy78xHi4DqxSSVFgJrDQo2fIbnIwmac/2a2v+3v830RmFzd76AdK4STRnejPeNpz806p60OWxTDiSzAuxW8syVqWsb4ygbu/4QAt1ohj9/bszverRLYM63TigFa64DilDtquOugkl7yJgQpKlk5ZvxyS6v3HDVVBakp9j8kOzvfvj8t3d1K+9T7B5YSGv28+EmOgls3NRjFLlQ379OUc9rqdF4YMHa7Whu+1sb+M+7b2ny86d3AhqR21B+UVRNcb0cmuwGnmjD7KU+RTLUk7+gpd5U7Dw6CiV8m+RbwbEHGvGfNQ4qhZikTP2ZgLV0DLNujaaf7/xbpRZvidTvhSzfgnm0gCd6sP1LiJSwq/4ItbpjpSxAa6YA+EHXgxvgrhzYVn0eymDVtoSVdaoYPP+q6GamgpXyoJfG86Kol31VkvR7NKKkUTroGGtiVrqkAc2WW166bT6dVzY6sn+Aiselo2TXoqCnZBfDC8ffdSO5R8g9b9ZDrNEwMKhJzAIg27Yy8mPcEXl9gqTbnODdkkm0XsvRf7baC2XgcOd+yvsf5C6D7d0zknT3zKE9FiH9bI0a6JRyWuqGmb2H8io9eqsanz+XE2oRU0kp53y22Q4s8uSkcBQDC909Su7r9Ze3bYW28hmbRp/0/1VmkBZZbNq7MwbVH6fJqPp9Z4kI58nbELco555kjF6IPWruIa81WHjCQY2uhiqxbPKTUx9TaCOg0GJUf5HFsd1+dcVx3X118f/C6Rw9/c="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()